In [9]:
import json
from uuid import uuid4
from datetime import datetime, timezone

current_time_str = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

In [2]:
file = json.load(open("../json/phi0830.phi001.edition_profile.json"))

In [2]:
uuid = uuid4()
print(f"urn:uuid:{uuid}")

urn:uuid:58fc6dc5-af16-47b6-80b8-2eb13fa20945


nundum G (corr. m1) P. 

In [10]:
current_time_str

'2026-10-25T20:17:09.690204Z'

In [3]:
template = json.load(open("../json/entry-claim-template.json"))

In [15]:
# Generate x apparatus claims
def generate_apparatus(num_claims, citation):
    entry = template[0]
    entry['type'] = "apparatus_entry"
    entry['id'] = "urn:uuid:" + str(uuid4())
    entry["text_locus"] = citation
    entry["source_edition"] = "urn:uuid:062e759d-619b-4549-b50b-e73f421609e3"
    entry["source_page"] = 5
    entry["source_image"] = "pdf-page-42"
    entry["transcriber"] = "SJH"
    entry["extraction"]["extracted_at"] = current_time_str
    claims = []
    for _ in range(num_claims):
        claim = template[1].copy()
        claim['type'] = "apparatus_claim"
        claim['id'] = "urn:uuid:" + str(uuid4())
        claim["apparatus_entry_id"] = entry['id']
        claim["target"] = citation
        claim["confidence"]["interpretation"] = "high"
        claim["confidence"]["reasons"] = "Text and witness sigla are clearly legible."
        claim["evaluation"]["reviewer"] = "SJH"
        claim["evaluation"]["review_status"] = "reviewed"
        claims.append(claim)
    cts = citation[17:]

    path = f"../json/{cts}.json"
    with open(path, "w") as f:
        json.dump([entry, *claims], f, indent=4)
        return (f"Generated {num_claims} apparatus claims for {citation} and saved to {path}")
        

In [21]:
generated_claims = generate_apparatus(
    6,
    "urn:cts:latinLit:phi0830.phi001.dll:1.13@Leuce",
)
print(generated_claims)

Generated 6 apparatus claims for urn:cts:latinLit:phi0830.phi001.dll:1.13@Leuce and saved to ../json/phi0830.phi001.dll:1.13@Leuce.json


In [ ]:
wit_list = ["κ", "α","χ", "edd."]

# Print the items in wit_list in order, with commas between them, and no comma after the last item
for wit in file["sigla"]:
    if wit["siglum"] in wit_list:
        print("{")
        print(f'"id": "{wit["uuid"]}",')
        print(f'"label": "{wit["siglum"]}"')
        # If the witness is the last in the list, don't print a comma after the label
        if wit["siglum"] != wit_list[-1]:
            print("},")
        else:
            print("}")

In [36]:
for wit in wit_list:
    if wit not in [w["siglum"] for w in file["sigla"]]:
        print(f"Warning: Witness '{wit}' not found in sigla list.")
    else:
        wit_data = next(w for w in file["sigla"] if w["siglum"] == wit)
        print("{")
        print(f'"id": "{wit_data["uuid"]}",')
        print(f'"label": "{wit_data["siglum"]}"')
        # If the witness is the last in the list, don't print a comma after the label
        if wit != wit_list[-1]:
            print("},")
        else:
            print("}")

{
"id": "urn:uuid:52fb95de-bce4-4c15-9e6d-b20967e32933",
"label": "κ"
},
{
"id": "urn:uuid:c295fc9f-988d-46e9-a43b-1d880157ca45",
"label": "α"
},
{
"id": "urn:uuid:8470035d-4a05-4b42-8cbc-98a14fa3f41d",
"label": "χ"
},


In [59]:
# add space between characters in the witness sigla for better readability
" ".join("βνκλγμπρ1φαηr").split()

['β', 'ν', 'κ', 'λ', 'γ', 'μ', 'π', 'ρ', '1', 'φ', 'α', 'η', 'r']

In [23]:
from pathlib import Path
def add_editorial_comments_to_json_files(json_dir):
    json_dir_path = Path(json_dir)

    for json_file in json_dir_path.glob("*.json"):
        with json_file.open("r+") as f:
            data = json.load(f)

            for obj in data:
                if obj.get("type") != "apparatus_claim":
                    continue

                if "editorial_comments" not in obj:
                    obj["editorial_comments"] = []

                if "evaluation" in obj:
                    evaluation = obj.pop("evaluation")
                    editorial_comments = obj.pop("editorial_comments")

                    obj["editorial_comments"] = editorial_comments
                    obj["evaluation"] = evaluation

            f.seek(0)
            json.dump(data, f, indent=4, ensure_ascii=False)
            f.truncate()

In [22]:
json_directory = "../json_copy"
add_editorial_comments_to_json_files(json_directory)

In [20]:
from pathlib import Path
import json

json_dir = Path("../json_copy")

for json_file in json_dir.glob("*.json"):
    with json_file.open() as f:
        data = json.load(f)

    for i, obj in enumerate(data):
        if not isinstance(obj, dict):
            print(json_file.name, i, type(obj).__name__)

phi0830.phi001.dll:1.5@explicuere.json 1 list


In [34]:
len("urn:cts:latinLit:")

17